# TechArena CTR pipeline
This notebook documents the leakage-safe workflow. The modules are the executable source of truth.

## 1. Configuration and dependencies
The default input files are in `/Users/monishobaid/Downloads`. Every statistic is built before a timestamp boundary `B`.

In [ ]:
from src.config import TRAIN_PATH, EVAL_PATH, CUT
print(TRAIN_PATH, EVAL_PATH, CUT)

## 2. Fake-data gate
The test generator uses the same JSON history schema and a no-click softmax. The test mutates future labels and verifies that earlier statistics do not change.

In [ ]:
from tests.test_pipeline import test_fake_pipeline
test_fake_pipeline()

## 3. Compact event cache
`src.build_cache` streams the raw CSV and writes candidate, label, history, timestamp and row ID arrays. Train and evaluation caches are recoded into one global user/item space.

In [ ]:
from src.cache import load_cache
from src.dataset import join_global
data = join_global(load_cache("cache/train.npz"), load_cache("cache/evaluation.npz"))
print(data.train.n_events, data.evaluation.n_events, data.n_users, data.n_items)

## 4. Phase 1 and Phase 2 features
The model combines rank-11 and rank-32 normalized click SVD, popularity-axis-removed taste scores, impression SVD, exact-history/recency features, item and user propensity, transitions, and event-level score summaries.

In [ ]:
# Reproduce the validation runner from the command line:
# python -m src.run_phase1 --rank 11 --alt-rank 32

## 5. No-click choice features
The generator-style model fits candidate utilities and an explicit no-click utility. Its outputs are supplied to LightGBM, where interactions are learned.

In [ ]:
# python -m src.run_choice_phase
# python -m src.predict

## 6. Submission checks
The final output is checked for exact row IDs, count, uniqueness, finite values, and probability bounds before writing CSV, ZIP and Excel artifacts.

In [ ]:
import json
print(json.load(open("outputs/submission_checks.json")))

## Phase A: local-to-portal gap diagnosis
The submitted model is evaluated at the ordinary Jan-26 first-event boundary, with history/gap bucket reports, support-aware reweighting, long-gap Jan-24 validation and adversarial validation. The decision metric is long-gap reweighted AUC because the ordinary boundary has no 1–3 day or 3–7 day validation support.

In [ ]:
# python -m src.phase_a
import json
print(json.load(open("outputs/phase_a_gap_diagnosis.json")))

## Phase B: synthetic twin
The twin preserves the real candidate slates and user structure, simulates labels from scaled rank-11 SVD utility plus popularity, repeat-click boost and a no-click option, then compares oracle, estimated SVD and full-pipeline AUC. Phase C remains gated until this ceiling test is reviewed.

In [ ]:
# python -m src.phase_b_twin
import json
print(json.load(open("outputs/phase_b_twin_results.json")))

## Phase A2/A3 review
A2 uses fixed three-day rates, popularity shares, percentiles, last-50-impression CTR, staleness and normalized impression cosine. DM23 and DM24 are evaluated with the same evaluation-matched reweighting. The half-day weighted A3 refit was attempted but exceeded the laptop runtime before a validated metric.

In [ ]:
# python -m src.phase_a2
# python -m src.phase_a2_safe
import json
print(json.load(open("outputs/phase_a2_results.json")))

## Phase B2 twin calibration
A scale/popularity grid was searched for a rank-11 SVD AUC near 0.79 while matching no-click and repeat CTR. No grid setting reached the SVD gate, so the twin was dropped as a test bench.

In [ ]:
# python -m src.phase_b2_calibrate
import json
print(json.load(open("outputs/phase_b2_calibration.json"))["best_by_svd"][:3])

## Phase C: real-data light-user experiments
The twin and adversarial diagnostics are retired. DM_mean is the mean of the evaluation-matched long-gap folds at B=23 Jan and B=24 Jan. The original production feature path is the reference. Experiment mode uses event-coherent 25% negative-event downsampling, LightGBM learning rate 0.1, 63 leaves and max_bin 127.

A3 DM_mean was 0.792807. C1 shrinkage plus a global no-click choice feature scored 0.790130, C2 added uncertainty features but was identical at 0.790130, and bounded recent-history EASE scored 0.787933. None clears the +0.003 gate against the original production DM_mean 0.797383, so Phase D/E are intentionally stopped.

In [ ]:
import json
from pathlib import Path
report=json.loads(Path('outputs/phase_c_real_results.json').read_text())
{k: report[k]['DM_mean'] for k in ['A3','base','c1','c2','c3']}


In [ ]:
import pandas as pd
pd.read_excel('outputs/phase_c_experiment_report.xlsx', sheet_name='summary')


## Phase C: real-data light-user experiments
The twin and adversarial diagnostics are retired. DM_mean is the mean of the evaluation-matched long-gap folds at B=23 Jan and B=24 Jan. The original production feature path is the reference. Experiment mode uses event-coherent 25% negative-event downsampling, LightGBM learning rate 0.1, 63 leaves and max_bin 127.

A3 DM_mean was 0.792807. C1 shrinkage plus a global no-click choice feature scored 0.790130, C2 added uncertainty features but was identical at 0.790130, and bounded recent-history EASE scored 0.787933. None clears the +0.003 gate against the original production DM_mean 0.797383, so Phase D/E are intentionally stopped.

In [ ]:
import json
from pathlib import Path
report=json.loads(Path('outputs/phase_c_real_results.json').read_text())
{k: report[k]['DM_mean'] for k in ['A3','base','c1','c2','c3']}


In [ ]:
import pandas as pd
pd.read_excel('outputs/phase_c_experiment_report.xlsx', sheet_name='summary')


## Corrected Phase C gate and final-mode result
The previous comparison used production DM24 as if it were DM_mean. The corrected experiment baseline is DM_mean=0.788117. C2 was corrected: C1 has shrink_dot, shrink_cos and shrink_norm; C2 adds posterior_std and posterior_utility_std.

A3+C1+C2 reaches experiment DM_mean=0.795972 (+0.007855). Full-data production is DM_mean=0.791842; the final three-seed lr=0.03 model reaches DM_mean=0.800742 (+0.008900), so SAFE_v2 was built and mechanically checked.

In [ ]:
import pandas as pd
pd.read_csv('outputs/phase_c_revision_summary.csv')


In [ ]:
import json
json.load(open('outputs/SAFE_v2_checks.json'))


## Phase V3 three-fold audit
DM22 was added at 2025-01-22. SAFE_v2 baseline experiment DM_mean3 is 0.790256; full-mode baseline is 0.795824. Actual 15-bootstrap taste bags, rank/time-decayed SVD views, and gap-aware item features all failed the +0.002 experiment gate.

Available diversity diagnostics were also below the baseline. CatBoost was unavailable in the offline runtime. No SAFE_v3/BEST_v3 was created.

In [ ]:
import pandas as pd
pd.read_csv('outputs/phase_v3_summary.csv')


## GPU SlateNet pipeline
`src/slatenet.py` is the supervised all-impression choice network. It requires the DCU GPU environment created by `scripts/setup_server.sh`; local execution is intentionally not attempted because torch is not installed in the laptop runtime and the configured GPU host was not DNS reachable. The job runs the fake-data leakage test first, then DM22/DM23/DM24 with sigmoid and softmax-choice outputs.

In [ ]:
# On the GPU host:
# bash scripts/setup_server.sh
# sbatch scripts/job_slatenet.sbatch


## TechArena handover continuation (2026-10-25)

The cached SAFE_v2 feature path was audited on DM22/DM23/DM24. The decomposition and two-stage diagnostics are saved in `outputs/decomposition_report.md`, `outputs/two_stage_report.md`, and `outputs/handover_results.json`. The event x choice product failed the SAFE gate; the best diagnostic logit blend retained 90% of the baseline and also failed the gate. No portal upload was made.

Forensics are in `outputs/forensics_report.md`; the SlateNet hand-set/L2 ladder and rejection are in `outputs/ladder_report.md`. The reproducible runner is `src/resume_handover.py`.

## Event propensity continuation (2026-10-25)

Corrected event model E uses the event label `(sum(clicked over the ten candidates) > 0)`, first-event fold alignment, and event-level history/gap reweighting. It reaches reweighted event AUC 0.6600/0.6674/0.6778 on DM22/DM23/DM24 (mean 0.6684).

User-rate persistence is 0.217 overall and 0.369 for users with 10+ events. E features, stacking, product, and the trained L0/L1/L2 fallback ladder are in `outputs/event_propensity_report.md` and `outputs/event_model_debug.md`. The best row-level blend is DM_mean3 0.790869, below the 0.793256 experiment gate; no full-mode run or `EVENT_STACK.csv` was created.

## Research-informed scorer bake-off

Priority 1 transductive slate statistics were allowed by the supplied competition instructions because they use unlabelled evaluation candidates only. Priority 2 restricted KL-NMF and Priority 3 OTTO-style co-visitation were evaluated alongside SVD/history references. No scorer cleared the residual-gain threshold; the best non-reference score was S8 surprise at DM24 0.675812, while CV_recent reached 0.531399. No BAKEOFF_STACK or BAKEOFF_BLEND file was created.

Reports: `outputs/bakeoff_report.md` and `outputs/artifact_report.md`; reproducible runner: `src/bakeoff_revised.py`.

## Taste-estimation bake-off setup check

The laptop environment has no `torch` or `gensim` in Python 3.12/3.13. Hardware is an 8 GB Apple M1 with 8 cores. The requested Torch/Gensim taste bake-off was therefore blocked before model fitting; see `outputs/taste_bakeoff_report.md`. The existing final-file recommendation is SAFE_v2 plus BLEND_SAFE_OLD.

## Taste bake-off CPU probe

The project `.venv` now imports Torch 2.14.1, Gensim 4.4.0, and LightGBM 4.7.0, but MPS reports unavailable. DM24 CPU probes on all 50,000 items gave NMF K8 0.540479, LDA K8 0.536556, and item2vec max cosine 0.553912; one-epoch SlateNet gave 0.583526. None is a taste win; see `outputs/taste_bakeoff_report.md`.

## Harness bug fix (2026-10-25)

The decisive DM24 self-test rebuilt rank-11 SVD inside the bake-off harness using the canonical `click_log_before` and reproduced cached `svd_cos` exactly: weighted AUC 0.773912415, delta 0, Spearman 1.0. The old bake-off scorers had used labelled train impressions only, omitting warm-up/history clicks. Corrected all-item CPU results are recorded in `outputs/harness_bug_report.md` and `outputs/taste_fixed_probe.json`; MPS is built but unavailable on this machine. No scorer cleared a taste-win or stack gate, so no submission file was produced.

## Final modelling attempt

Implemented rank-11 logistic MF with C1/SVD initialization and batched IRLS/Newton updates, plus verified SlateNet L3 with trainable item embeddings and SVD anchor penalties. Logistic MF lambda=100 cosine reached DM24 0.773996; lambda=10 cosine residual gain was +0.002326 on DM24 but only +0.001782 across adjacent-fold mean3 confirmation. SlateNet L3 stayed below frozen L2. No taste win or TASTE_STACK.csv. See `outputs/final_summary.md`.

## Focused event/listwise attempt

A grouped 11-choice LightGBM LambdaRank model with ten candidate rows plus an explicit no-click row, strict-before-B session features, slate aggregates and SAFE_v2 features reached DM22 0.775297, DM23 0.784617, DM24 0.794414, mean3 0.784776. It did not close the SAFE_v2 gap and was rejected.

## Protocol experiment Stage 1

R1 (B-aligned cross-fitted taste plus all events) scored DM24 0.780527; R2 (same features plus first-event windows) scored 0.780355; R0 SAFE_v2 full reference is 0.805522 on DM24 and 0.795824 mean3. Both failed the +0.002 DM24 trigger, so later stages were not run. See `outputs/protocol_report.md`.

## Protocol experiment Stage 1

R1 (B-aligned cross-fitted taste plus all events) scored DM24 0.780527; R2 (same features plus first-event windows) scored 0.780355; R0 SAFE_v2 full reference is 0.805522 on DM24 and 0.795824 mean3. Both failed the +0.002 DM24 trigger, so later stages were not run. See `outputs/protocol_report.md`.

## SAFE_LOGMF10 follow-up

The supplied graph80 file passed structural checks; its Spearman correlation with SAFE_LOGMF10 was 0.852738. BLEND_LMF_GRAPH85.csv and BLEND_LMF_GRAPH70.csv were written. Logistic-MF variants lambda=100, rank-16 lambda=10, and recent-3-day lambda=10 all failed the +0.001 DM24 gate. Five bounded LightGBM trials selected a parameter set with experiment DM_mean3 0.790834. Seven-seed full confirmation reached DM22 0.783848, DM23 0.792948, DM24 0.802374, mean3 0.793057, missing the SAFE_LOGMF10 full gate by 0.000034; SAFE_LOGMF_v2.csv was not written. See outputs/lmf_followup_report.md.

## Graph-signal diagnosis and fold-safe graph stack
Graph80 forensics and the boundary-safe graph scorer audit are in `outputs/graph_stack_report.md`. The standalone graph family stayed near 0.52–0.56 DM AUC and the SAFE_LOGMF10 residual gate failed (+0.002), so no graph stack submission was written.


## Next-step modelling audit
The LambdaRank, candidate-conditioned SVD attention, and all-event protocol diagnostics are summarized in `outputs/next_step_modelling_report.md`. None passed the promotion gate.


## Graph80 recovery and conditional blends
The recovered graph80 artifact and two regime-conditional portal candidates are in `outputs/graph_recovery_report.md`. Source code was not found; no upload was performed.


## Stronger conditional graph blends
Two stronger portal candidates are in `outputs/stronger_conditional_graph_report.md`; both passed row/schema checks and were not uploaded.


## Event-adaptive graph blend
The event-adaptive candidate is in `outputs/event_adaptive_graph_report.md`; source recovery remained unavailable, and the candidate was not uploaded.
